# 01 Basket Diagnostics

Build market-basket-ready item tables at category, handle, and SKU/flavor levels from cleaned EDA outputs.

In [1]:
from pathlib import Path
import sys


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "EDA" / "outputs").exists() and (candidate / "product_mba").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root containing EDA/outputs and product_mba")

PROJECT_ROOT = find_project_root()
EDA_OUTPUTS = PROJECT_ROOT / "EDA" / "outputs_finals"  # FINALS cohort
MBA_DIR = PROJECT_ROOT / "product_mba"
MBA_OUTPUTS = MBA_DIR / "outputs"
MBA_OUTPUTS.mkdir(exist_ok=True)

print("Project root found")
print("EDA outputs: EDA/outputs_finals (finals cohort)")
print("MBA outputs: product_mba/outputs")


Project root found
EDA outputs: EDA/outputs_finals (finals cohort)
MBA outputs: product_mba/outputs


In [2]:
import re
import numpy as np
import pandas as pd

lines = pd.read_parquet(EDA_OUTPUTS / "lines_sku_analysis.parquet")
orders = pd.read_parquet(EDA_OUTPUTS / "orders.parquet")
customers = pd.read_parquet(EDA_OUTPUTS / "customers.parquet")

lines["order_id"] = lines["order_id"].astype(str)
orders["order_id"] = orders["order_id"].astype(str)
lines["order_date"] = pd.to_datetime(lines["order_date"], utc=True)

def normalize_sku(value):
    if pd.isna(value):
        return pd.NA
    text = str(value).strip().strip("'").strip('"').strip()
    if not text or text.lower() in {"nan", "none", "null"}:
        return pd.NA
    if re.fullmatch(r"\d+\.0", text):
        text = text[:-2]
    return text.upper()

def clean_label(value):
    if pd.isna(value):
        return pd.NA
    text = str(value).strip()
    if not text or text.lower() in {"nan", "none", "null", "unknown"}:
        return pd.NA
    return text

lines["sku_norm"] = lines["Line: SKU"].map(normalize_sku)
lines["category_item"] = lines["product_category"].map(clean_label)
lines["handle_item"] = lines["Line: Product Handle"].map(clean_label)
lines["title_clean"] = lines["Line: Title"].map(clean_label)
lines["variant_clean"] = lines["Line: Variant Title"].map(clean_label)
lines["sku_item"] = np.where(
    lines["sku_norm"].notna(),
    lines["sku_norm"].astype(str) + " | " + lines["title_clean"].fillna("Untitled") + " | " + lines["variant_clean"].fillna("Default"),
    pd.NA,
)

print("Loaded", len(lines), "line items across", lines["order_id"].nunique(), "orders")


Loaded 14448 line items across 8955 orders


In [3]:
def basket_diagnostics(df, item_col, label, actionable=True):
    item_df = df[["order_id", "customer_id", "order_date", item_col]].dropna(subset=[item_col]).drop_duplicates(["order_id", item_col])
    basket_size = item_df.groupby("order_id")[item_col].nunique()
    total_orders = df["order_id"].nunique()
    orders_with_item = basket_size.size
    summary = {
        "level": label,
        "total_orders": total_orders,
        "orders_with_any_item": orders_with_item,
        "analysis_ready_order_pct": orders_with_item / total_orders if total_orders else np.nan,
        "avg_distinct_items": basket_size.mean(),
        "median_distinct_items": basket_size.median(),
        "p90_distinct_items": basket_size.quantile(0.90) if len(basket_size) else np.nan,
        "multi_item_orders": int((basket_size >= 2).sum()),
        "multi_item_order_pct": (basket_size >= 2).mean() if len(basket_size) else np.nan,
        "three_plus_item_orders": int((basket_size >= 3).sum()),
        "three_plus_item_order_pct": (basket_size >= 3).mean() if len(basket_size) else np.nan,
        "unique_items": item_df[item_col].nunique(),
        "actionable": actionable,
    }
    top_items = (
        item_df.groupby(item_col)
        .agg(orders=("order_id", "nunique"), customers=("customer_id", "nunique"))
        .reset_index()
        .sort_values("orders", ascending=False)
    )
    basket_size_dist = basket_size.value_counts().sort_index().rename_axis("distinct_items").reset_index(name="orders")
    return item_df, summary, top_items, basket_size_dist

levels = [
    ("category_item", "category", False),
    ("handle_item", "handle", True),
    ("sku_item", "sku_flavor", True),
]

summaries = []
for item_col, label, actionable in levels:
    item_df, summary, top_items, basket_size_dist = basket_diagnostics(lines, item_col, label, actionable)
    summaries.append(summary)
    item_df.to_parquet(MBA_OUTPUTS / f"basket_items_{label}.parquet", index=False)
    top_items.to_csv(MBA_OUTPUTS / f"basket_top_items_{label}.csv", index=False)
    basket_size_dist.to_csv(MBA_OUTPUTS / f"basket_size_distribution_{label}.csv", index=False)

missing_summary = pd.DataFrame([
    {"field": "Line: Product Handle", "missing_pct": lines["Line: Product Handle"].isna().mean(), "nonnull_unique": lines["Line: Product Handle"].nunique(dropna=True)},
    {"field": "Line: SKU", "missing_pct": lines["Line: SKU"].isna().mean(), "nonnull_unique": lines["Line: SKU"].nunique(dropna=True)},
    {"field": "product_category == Unknown", "missing_pct": (lines["product_category"] == "Unknown").mean(), "nonnull_unique": lines["product_category"].nunique(dropna=True)},
])
summary_df = pd.DataFrame(summaries)
summary_df.to_csv(MBA_OUTPUTS / "basket_diagnostics_summary.csv", index=False)
missing_summary.to_csv(MBA_OUTPUTS / "basket_missingness_summary.csv", index=False)

print("Basket diagnostics summary")
display(summary_df)
print("\nMissingness summary")
display(missing_summary)
for label in ["category", "handle", "sku_flavor"]:
    print(f"\nTop items: {label}")
    display(pd.read_csv(MBA_OUTPUTS / f"basket_top_items_{label}.csv").head(10))


Basket diagnostics summary


,level,total_orders,orders_with_any_item,analysis_ready_order_pct,avg_distinct_items,median_distinct_items,p90_distinct_items,multi_item_orders,multi_item_order_pct,three_plus_item_orders,three_plus_item_order_pct,unique_items,actionable
0,category,8955,7347,0.820436,1.293589,1.0,2.0,1780,0.242276,336,0.045733,6,False
1,handle,8955,7347,0.820436,1.347897,1.0,2.0,1979,0.269362,485,0.066013,24,True
2,sku_flavor,8955,7564,0.844668,1.585140,1.0,3.0,2898,0.383131,1177,0.155605,363,True



Missingness summary


,field,missing_pct,nonnull_unique
0,Line: Product Handle,0.232350,24
1,Line: SKU,0.165213,157
2,product_category == Unknown,0.232350,7



Top items: category


,category_item,orders,customers
0,Other,3082,2046
1,Clear Protein,2130,1614
2,Lean Protein,1939,1395
3,Accessories,1068,990
4,Collagen Glow,839,516
5,Soy Protein,446,320



Top items: handle


,handle_item,orders,customers
0,clear-protein,1868,1404
1,lean-protein,1582,1145
2,better-whey,1199,840
3,lushprotein-clear-shaker,1068,990
4,collagen-glow,820,503
5,plant-protein,632,422
6,micronized-creatine-monohydrate,623,470
7,soy-protein-isolate,446,320
8,lushprotein-lean-protein-40g-single-serve,322,275
9,prime-whey-isolate,306,191



Top items: sku_flavor


,sku_item,orders,customers
0,0724999807814 | CLEAR PROTEIN | 500g Pack (20 ...,654,532
1,0724999808361 | LP CLASSIC SHAKER | White,473,451
2,0724999807807 | CLEAR PROTEIN | 500g Pack (20 ...,424,362
3,ACC-SHK-V2 | LP CLASSIC SHAKER | White,341,316
4,LEAN-THA-1KG-V1 | LEAN PROTEIN | 1 x 1kg Pack ...,336,275
5,0724999807821 | LEAN PROTEIN | 1kg Pack (25 se...,283,237
6,CLEAR-PEA-500G-V2 | CLEAR PROTEIN | 1 x 500g P...,273,237
7,LEAN-TAR-1KG-V1 | LEAN PROTEIN | 1 x 1kg Pack ...,259,205
8,CLEAR-GRA-500G-V2 | CLEAR PROTEIN | 1 x 500g P...,244,207
9,CRE-UNF-250G-V1 | CREATINE MONOHYDRATE | 1 x 2...,232,192
